# L1 SCENIC regulon vs DEG hypergeometric enrichment
Preprocessing kept to the minimum:
1. parse `TargetGenes` from the L1 SCENIC ctx results (`/data1st1/junyi/output/sn0827L1/scenic_ctx/`)
   into a TF -> target-gene regulon table per region x L1 celltype (only TFs with any motif NES > 3);
2. load the L1 MAST DEG table and keep FDR < 0.05, split by Up/Down;
3. for every celltype x condition x TF, hypergeometric test of regulon targets against
   up/down DEGs (`hypergeom.sf(k-1, M, n, N)`), BH-corrected, saved to the L1 output folder.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import glob
import os
from ast import literal_eval

import numpy as np
import pandas as pd
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests

In [2]:
# paths
ctx_dir = "/data1st1/junyi/output/sn0827L1/scenic_ctx/"
regulon_dir = "/data1st1/junyi/output/sn0827L1/tf_regulon/"
out_dir = "/data1st1/junyi/output/sn0827L1/tf_regulon_deg_hypergeom/"
deg_file = "/data2st2/junyi/output/hannan25_L1_mast_degs_fdr_merged_4conditions_7regions.csv"

NES_THRESHOLD = 3.0   # keep TargetGenes of motifs with NES > threshold
DEG_FDR = 0.05        # DEG significance cut-off
os.makedirs(regulon_dir, exist_ok=True)
os.makedirs(out_dir, exist_ok=True)

In [3]:
# preprocessing: ctx csv -> TF target-gene table per region x L1 celltype
# one TF can be represented by several motifs; targets of its significant motifs are unioned
def parse_enrichment_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return literal_eval(value)

regulon = {}  # fname -> {TF: set(target genes)}
for f in sorted(glob.glob(os.path.join(ctx_dir, "ctx_*.csv"))):
    fname = os.path.basename(f).replace("ctx_", "").replace(".csv", "")
    df = pd.read_csv(f)
    # ctx csv has a multi-row header: Enrichment.1 = NES, Enrichment.6 = TargetGenes
    df = df.rename(columns={"Unnamed: 0": "TF", "Unnamed: 1": "motif",
                            "Enrichment.1": "NES", "Enrichment.6": "TargetGenes"})
    df = df.dropna(subset=["TargetGenes"]).copy()
    df["NES"] = pd.to_numeric(df["NES"], errors="coerce")
    df = df[df["NES"] > NES_THRESHOLD]
    tf_targets = {}
    for tf, tg in zip(df["TF"], df["TargetGenes"]):
        genes = {g for g, _ in parse_enrichment_list(tg)}
        tf_targets.setdefault(tf, set()).update(genes)
    if not tf_targets:
        print(f"skip {fname}: no regulon with NES > {NES_THRESHOLD}")
        continue
    regulon[fname] = tf_targets
    pd.DataFrame({"TF": list(tf_targets),
                  "n_target": [len(v) for v in tf_targets.values()],
                  "targetgene": [",".join(sorted(v)) for v in tf_targets.values()]}
                 ).to_csv(os.path.join(regulon_dir, f"{fname}_tf_target.csv"), index=False)

print(f"{len(regulon)} cell types, e.g. {list(regulon)[:3]}")

68 cell types, e.g. ['AMY_Astrocyte', 'AMY_Epen', 'AMY_GABA']


In [4]:
# load DEG table (L1 MAST, merged over 4 conditions x 7 regions), keep FDR < 0.05
df_deg = pd.read_csv(deg_file)
df_deg = df_deg[(df_deg["FDR"] < DEG_FDR) & (df_deg["log2FC"].abs() > 0.1)].copy()
df_deg["ctname"] = df_deg["Region"].astype(str) + "_" + df_deg["Class"].astype(str)
# compare gene symbols case-insensitively (ctx targets and DEG genes may differ in case)
df_deg["gene_up"] = df_deg["Gene"].astype(str).str.strip().str.upper()
df_deg = df_deg.drop_duplicates(["ctname", "Model", "gene_up"])
print("DEG rows:", len(df_deg), "| celltypes:", df_deg.ctname.nunique(), "| models:", sorted(df_deg.Model.unique()))

# DEG gene sets per celltype x Model x Direction
deg_sets = {
    (ct, model, d): set(g)
    for (ct, model, d), g in df_deg.groupby(["ctname", "Model", "Direction"])["gene_up"]
}

DEG rows: 173959 | celltypes: 64 | models: ['CSRES M', 'CSSUS M', 'CURES F', 'CURES M', 'CUSUS F', 'CUSUS M']


In [5]:
df_deg

,Gene,pval,log2FC,ci.hi,ci.lo,FDR,Bonferroni,celltype.L2,Subclass,Comparison,...,DGL3,Class,mean_UMI_raw_Region_Class,mean_UMI_norm_Region_Class,mean_log1p_Region_Class,mean_UMI_raw_Region_subclass,mean_UMI_norm_Region_subclass,mean_log1p_Region_subclass,ctname,gene_up
3,9330175M20Rik,0.000000e+00,0.234866,0.245376,0.224356,0.000000e+00,0.000000e+00,AMY Glut,AMY Glut,CSRES_vs_CON,...,NaN,Glut,NaN,NaN,NaN,NaN,NaN,NaN,AMY_Glut,9330175M20RIK
4,A830018L16Rik,1.491574e-80,0.175274,0.193258,0.157290,4.460835e-80,1.150003e-76,AMY Glut,AMY Glut,CSRES_vs_CON,...,NaN,Glut,NaN,NaN,NaN,NaN,NaN,NaN,AMY_Glut,A830018L16RIK
7,Acsl3,1.567864e-316,0.206751,0.217397,0.196104,2.864510e-315,1.208823e-312,AMY Glut,AMY Glut,CSRES_vs_CON,...,NaN,Glut,NaN,NaN,NaN,NaN,NaN,NaN,AMY_Glut,ACSL3
8,Actr1b,0.000000e+00,-0.267039,-0.253994,-0.280085,0.000000e+00,0.000000e+00,AMY Glut,AMY Glut,CSRES_vs_CON,...,NaN,Glut,NaN,NaN,NaN,NaN,NaN,NaN,AMY_Glut,ACTR1B
10,Adgrb3,5.787503e-198,0.132657,0.141236,0.124078,5.231143e-197,4.462165e-194,AMY Glut,AMY Glut,CSRES_vs_CON,...,NaN,Glut,NaN,NaN,NaN,NaN,NaN,NaN,AMY_Glut,ADGRB3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
583314,Gm15155,6.000726e-05,0.119395,0.219762,0.019029,1.605083e-03,4.333725e-01,TH GABA,TH GABA,SUS_vs_CON,...,NaN,GABA,NaN,NaN,NaN,NaN,NaN,NaN,TH_GABA,GM15155
583315,Htr2c,1.062084e-10,-0.301904,-0.209815,-0.393994,1.783808e-08,7.670373e-07,TH GABA,TH GABA,SUS_vs_CON,...,NaN,GABA,NaN,NaN,NaN,NaN,NaN,NaN,TH_GABA,HTR2C
583317,Pak3,1.548961e-05,-0.217542,-0.126778,-0.308306,5.352438e-04,1.118660e-01,TH GABA,TH GABA,SUS_vs_CON,...,NaN,GABA,NaN,NaN,NaN,NaN,NaN,NaN,TH_GABA,PAK3
583318,Pcdh11x,2.179373e-05,0.273971,0.394549,0.153393,7.154289e-04,1.573943e-01,TH GABA,TH GABA,SUS_vs_CON,...,NaN,GABA,NaN,NaN,NaN,NaN,NaN,NaN,TH_GABA,PCDH11X


In [6]:
# hypergeometric test: TF regulon targets vs up/down DEGs for each celltype x Model
#   M = background = union(regulon targets of this celltype, DEG genes of this celltype+Model)
#   n = regulon size, N = n up (or down) DEGs, k = overlap
rows = []
for fname, tf_targets in regulon.items():
    ct_deg = df_deg[df_deg.ctname == fname]
    if ct_deg.empty:
        print(f"skip {fname}: no DEGs")
        continue
    for (ct, model), g in ct_deg.groupby(["ctname", "Model"]):
        deg_up = deg_sets.get((ct, model, "Up"), set())
        deg_down = deg_sets.get((ct, model, "Down"), set())
        background = set().union(*tf_targets.values()) | deg_up | deg_down
        M = len(background)
        for tf, targets in tf_targets.items():
            tset = {gene.upper() for gene in targets}
            n = len(tset)
            k_up = len(tset & deg_up)
            k_down = len(tset & deg_down)
            rows.append({
                "ctname": ct, "Region": ct.split("_")[0], "Celltype": ct.split("_", 1)[1],
                "Model": model, "TF": tf, "n_target": n,
                "n_deg_up": len(deg_up), "n_deg_down": len(deg_down),
                "n_overlap_up": k_up, "n_overlap_down": k_down,
                "hypergeom_p_up": hypergeom.sf(k_up - 1, M, n, len(deg_up)) if k_up > 0 else 1.0,
                "hypergeom_p_down": hypergeom.sf(k_down - 1, M, n, len(deg_down)) if k_down > 0 else 1.0,
                "background_size": M,
                "overlap_up_genes": ",".join(sorted(tset & deg_up)),
                "overlap_down_genes": ",".join(sorted(tset & deg_down)),
            })

df_hyp = pd.DataFrame(rows)
df_hyp["hypergeom_FDR_up"] = multipletests(df_hyp["hypergeom_p_up"], method="fdr_bh")[1]
df_hyp["hypergeom_FDR_down"] = multipletests(df_hyp["hypergeom_p_down"], method="fdr_bh")[1]
df_hyp = df_hyp.sort_values(["ctname", "Model", "hypergeom_p_up", "hypergeom_p_down"]).reset_index(drop=True)
print(df_hyp.shape)
df_hyp.head()

skip AMY_Immune: no DEGs
skip HPF_Immune: no DEGs
skip PFC_Epen: no DEGs
skip PFC_Immune: no DEGs
(68115, 17)


,ctname,Region,Celltype,Model,TF,n_target,n_deg_up,n_deg_down,n_overlap_up,n_overlap_down,hypergeom_p_up,hypergeom_p_down,background_size,overlap_up_genes,overlap_down_genes,hypergeom_FDR_up,hypergeom_FDR_down
0,AMY_Astrocyte,AMY,Astrocyte,CSRES M,Jund,95,629,622,49,2,1.517541e-23,0.999732,5862,"ACTB,APLP1,ATP1B1,ATP6V0C,BEX1,BEX2,BMYC,CACNA...","CASKIN1,PSAP",6.341552e-21,1.0
1,AMY_Astrocyte,AMY,Astrocyte,CSRES M,Nfe2l1,64,629,622,33,2,3.305983e-16,0.993641,5862,"ATP1A1,ATP1A3,ATP1B1,ATP6V0C,C1QL3,CAMK1D,CHGB...","ARHGEF11,GFAP",8.694480e-14,1.0
2,AMY_Astrocyte,AMY,Astrocyte,CSRES M,Npdc1,171,629,622,57,12,5.304547e-16,0.959498,5862,"A830018L16RIK,ACTB,APLP1,ARL6IP5,ATP1B1,ATP5G1...","ATP6V0B,CTSD,FAM173A,GLT8D1,GPRC5B,GRINA,MT3,N...",1.363469e-13,1.0
3,AMY_Astrocyte,AMY,Astrocyte,CSRES M,Nr4a1,104,629,622,31,2,6.056996e-08,0.999895,5862,"5730522E02RIK,ANKS1B,APP,CACNA1C,CAR10,CTNNA3,...","ASAP1,SRSF7",7.064594e-06,1.0
4,AMY_Astrocyte,AMY,Astrocyte,CSRES M,Junb,270,629,622,49,0,1.243557e-04,1.000000,5862,"5730522E02RIK,A830018L16RIK,APP,ATP1B1,ATP6AP2...",,8.036514e-03,1.0


In [7]:
# merge hypergeom results with
#   (a) the MAST DEG table  -> is the TF gene itself a DEG in this celltype/Model? its log2FC/FDR/Direction
#   (b) the L1 AUC wilcoxon -> the TF regulon's own AUC log2FC / FDR under wilcoxon
# comparison naming: AUC wilcoxon "SUS_M" etc. maps to MAST Model "CUSUS_M" etc.
comparison_to_model = {
    'SUS_M': 'CUSUS_M', 'SUS_F': 'CUSUS_F',
    'RES_M': 'CURES_M', 'RES_F': 'CURES_F',
    'CSDS_M': 'CSSUS_M', 'CSDS_F': 'CSSUS_F',
    'CSRES_M': 'CSRES_M',
}

df_hyp['TF_up'] = df_hyp['TF'].str.upper()
df_hyp['TF'] = df_hyp['TF'].str.lower().str.capitalize()
df_hyp['Model'] = df_hyp['Model'].str.replace(' ', '_')

# (a) TF gene itself in the DEG list (full MAST table, before FDR filtering)
df_deg_full = pd.read_csv(deg_file)
df_deg_full['ctname'] = df_deg_full['Region'].astype(str) + '_' + df_deg_full['Class'].astype(str)
df_deg_full['Model'] = df_deg_full['Model'].str.replace(' ', '_')
df_deg_full['TF_up'] = df_deg_full['Gene'].astype(str).str.strip().str.upper()
#df_deg_full = df_deg_full[(df_deg_full["FDR"] < DEG_FDR) & (df_deg_full["log2FC"].abs() > 0.1)].copy()
df_deg_info = (df_deg_full.drop_duplicates(['ctname', 'Model', 'TF_up'])
               [['ctname', 'Model', 'TF_up', 'log2FC', 'FDR', 'Direction']]
               .rename(columns={'log2FC': 'TF_DEG_log2FC', 'FDR': 'TF_DEG_FDR',
                                'Direction': 'TF_DEG_Direction'}))
df_hyp = df_hyp.merge(df_deg_info, on=['ctname', 'Model', 'TF_up'], how='left')
# TF itself counts as a DEG only if FDR < threshold AND |log2FC| > 0.1
DEG_LFC_CUTOFF = 0.1
df_hyp['Is_TF_DEG'] = (df_hyp['TF_DEG_FDR'].notna()
                       & (df_hyp['TF_DEG_FDR'] < DEG_FDR)
                       & (df_hyp['TF_DEG_log2FC'].abs() > DEG_LFC_CUTOFF))
df_hyp['Is_TF_DEG_up'] = df_hyp['Is_TF_DEG'] & (df_hyp['TF_DEG_Direction'] == 'Up')
df_hyp['Is_TF_DEG_down'] = df_hyp['Is_TF_DEG'] & (df_hyp['TF_DEG_Direction'] == 'Down')

# (b) TF regulon AUC change from the L1 AUC wilcoxon
df_auc = pd.read_csv("/data2/junyi/stg0901/scenic_wilcoxon_L1/combined_wilcoxon_DEGs.csv")
df_auc['TF_up'] = df_auc['TF'].str.replace("(+)", "", regex=False).str.upper()
df_auc['Model'] = df_auc['comparison'].map(comparison_to_model)
df_auc['ctname'] = df_auc['region'].astype(str) + '_' + df_auc['celltype.L1'].astype(str)
df_auc_info = (df_auc.drop_duplicates(['ctname', 'Model', 'TF_up'])
               [['ctname', 'Model', 'TF_up', 'log2FC', 'FDR', 'Direction']]
               .rename(columns={'log2FC': 'AUC_log2FC', 'FDR': 'AUC_FDR',
                                'Direction': 'AUC_Direction'}))
df_hyp = df_hyp.merge(df_auc_info, on=['ctname', 'Model', 'TF_up'], how='left')
df_hyp['Is_AUC_sig'] = df_hyp['AUC_FDR'].notna() & (df_hyp['AUC_FDR'] < 0.05)
df_hyp['Is_AUC_sig_up'] = df_hyp['Is_AUC_sig'] & (df_hyp['AUC_Direction'] == 'Up')
df_hyp['Is_AUC_sig_down'] = df_hyp['Is_AUC_sig'] & (df_hyp['AUC_Direction'] == 'Down')

# how often the TF itself is a DEG / its regulon AUC is differential, among significant tests
sig = df_hyp[(df_hyp.hypergeom_FDR_up < 0.05) | (df_hyp.hypergeom_FDR_down < 0.05)]
print(f"significant hypergeom tests: {len(sig)} | TF itself DEG: {sig.Is_TF_DEG.sum()} "
      f"| AUC sig: {sig.Is_AUC_sig.sum()}")
df_hyp.head()

significant hypergeom tests: 2615 | TF itself DEG: 879 | AUC sig: 2272


,ctname,Region,Celltype,Model,TF,n_target,n_deg_up,n_deg_down,n_overlap_up,n_overlap_down,...,TF_DEG_Direction,Is_TF_DEG,Is_TF_DEG_up,Is_TF_DEG_down,AUC_log2FC,AUC_FDR,AUC_Direction,Is_AUC_sig,Is_AUC_sig_up,Is_AUC_sig_down
0,AMY_Astrocyte,AMY,Astrocyte,CSRES_M,Jund,95,629,622,49,2,...,Up,False,False,False,0.580512,2.683045e-213,Up,True,True,False
1,AMY_Astrocyte,AMY,Astrocyte,CSRES_M,Nfe2l1,64,629,622,33,2,...,Up,True,True,False,1.182490,0.000000e+00,Up,True,True,False
2,AMY_Astrocyte,AMY,Astrocyte,CSRES_M,Npdc1,171,629,622,57,12,...,NaN,False,False,False,0.294799,2.061941e-123,Up,True,True,False
3,AMY_Astrocyte,AMY,Astrocyte,CSRES_M,Nr4a1,104,629,622,31,2,...,NaN,False,False,False,0.537215,1.048214e-228,Up,True,True,False
4,AMY_Astrocyte,AMY,Astrocyte,CSRES_M,Junb,270,629,622,49,0,...,NaN,False,False,False,0.606767,0.000000e+00,Up,True,True,False


In [8]:
df_hyp.to_csv(os.path.join(out_dir, "L1_TF_regulon_deg_hypergeom.csv"), index=False)
sig = df_hyp[(df_hyp.hypergeom_FDR_up < 0.05) | (df_hyp.hypergeom_FDR_down < 0.05)]
sig.to_csv(os.path.join(out_dir, "L1_TF_regulon_deg_hypergeom_FDR05.csv"), index=False)
print("saved:", out_dir)
print(f"all tests: {len(df_hyp)} | significant (FDR<0.05): {len(sig)}")

saved: /data1st1/junyi/output/sn0827L1/tf_regulon_deg_hypergeom/
all tests: 68115 | significant (FDR<0.05): 2615
